# 000_wfi_select_cals

This notebook is used to select datasets for calibration of WFI frames. 
The user should enter:

    - a directory where all the science data to be processed lies,
    - validity times for each of BIAS, DOME FLATS, and SKY_FLATS.
    - validity exposure times for sky flats

The script will output a list of all calibration frames needed by the mefpp 
data reduction engine.

Because WFGI does not have quality control Garching support it is not supported
by calSelector. Therefore, when data is retrieved from the archive no valid
calibration data can be attached. This script is design to help in the selection
of data. It cannot do QC on the data because it is not on disk yet. So, after
the script runs a notebook to do minimal quality controld on the calibration frames
should be run.

The QC note book will:

    - check level and dispersion of each bias and dome frames.
    - check exposure times of sky frames and only accept those with exptime < 60s (TBC)
    - compare sky frame levels with the prediction of models of
      sky brightness versus sun altitude. This will permit the determination of
      cloud presence.

In [ ]:
import glob
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) 
pd.set_option('display.max_rows', None)
from IPython.display import display
import numpy as np
from datetime import datetime, timedelta
from astropy.io import fits

In [ ]:
class ExecutionBlock_WFI:
    """
    Class to contain all exposures of a single template execution.
    It is constructed by passing as argument a list of exposures
    with the same TPL.START keyword. For BIAS exposures this is
    the only key that matters. For other DOME and SKY flat fields
    the exposures should have the same filter and (detector modes
    but these are not searchable in the ESO archive).
    """
    def __init__(self, exposure_list):
        self.exposure_list = exposure_list
        self.nexpo = len(exposure_list)
        self.tpl_start = exposure_list[0].tpl_start
        self.filter = exposure_list[0].filt

        self.exptime_min = 1000000.0
        self.exptime_max = -10000.0
        self.data_setid_list = []
        for expo in exposure_list:
            if expo.tpl_start != self.tpl_start:
                raise Exception("ExecutionBlock: discordant tpl_start")

            if expo.filt != self.filter:
                raise Exception("ExecutionBlock: discordant filter")

            if expo.exptime > self.exptime_max:
                self.exptime_max = expo.exptime

            if expo.exptime < self.exptime_min:
                self.exptime_min = expo.exptime 

            self.data_setid_list.append(expo.DatasetID)
        
class CalExp_WFI:
    """
    Class to contain information about a single exposure, from the archive.
    """
    def __init__(self, series):
        self.OBJECT = series['OBJECT']
        self.RA = series['RA']
        self.DEC = series['DEC']
        self.Program_ID = series['Program_ID']
        self.Instrument = series['Instrument']
        self.dpr_catg = series['Category']
        self.dpr_type = series['Type']
        self.mode = series['Mode']
        self.DatasetID = series['Dataset ID']
        self.release_date = series['Release_Date']
        self.tpl_id = series['TPL ID']
        self.tpl_start = series['TPL START']
        self.exptime = series['Exptime']
        self.filt_lbda_min = series['filter_lambda_min']
        self.filt_lbda_max = series['filter_lambda_max']
        self.filt = series['Filter']
        self.mjd_obs = series['MJD-OBS']
        self.airmass = series['Airmass']
        self.dimm_seeing = series['DIMM Seeing at Start']

def reform_df(df, time_key):
    # here we remove rows which do not contain a value of TPL.START
    # This may remove older data without that key.
    df = df[df[time_key].str.strip().astype(bool)]
    df.dropna(inplace=True)
    for index, row in df.iterrows():
        try:
            pd.to_datetime(row[time_key])
        except:
            print(index, row)

    # create a column with datetime dates
    df['Date'] = df.apply(lambda row: pd.to_datetime(row[time_key]), axis = 1)
    #df['Date'] = pd.to_datetime(df[time_key]) # .dt.strftime('%Y-%m-%dT%H:%M:%S'))
    return df

def select_calibrations(df, obs_date, filt, valid_days=10, nmin_in_block=1, nagg_min=1, cal_type='FLAT,SKY', time_key='TPL START', debug=True):
    """
    Given:
        - a pandas dataframe with a table of calibrations, 
        - obs_date: an observation date,
        - filt: a filter required for flat fields
        - valid_days: a number of validity days for a calibration
        - nmin_in_block: the minimum number of observations in a template execution
        - nagg_min: a minimum total number of frames of the type required,
        - cal_type: the type of calibration requested,
        - time_key: which table column contains the time stamp

        this procedure search the pandas table df for execution blocks taken within valid_days
        from obs_date. Then get all frames in the nearest blocks until it have
        nagg_min. If nagg_min is 1, then it returns all frames from a single execution
        block. In the case of WFI for which the archive does not give us OB.START keys
        we group by TPL.START.
    """

    start_date = obs_date - timedelta(days=valid_days)
    end_date = obs_date + timedelta(days=valid_days)

    # extract calibrations between those days
    filtered_df = df.loc[(start_date <= df['Date']) & (df['Date'] < end_date)]

    # we partition by TPL.START and FILTER
    n_by_tpl_start = filtered_df.groupby(["TPL START", "Filter"]) #.agg(["min", "max", "count"])

    # we create a groups object
    gb = n_by_tpl_start.groups

    # Here we access the rows of the pandas dataframe group by group
    execution_block_dict = {}
    for key, value in gb.items():    
        if key[1] == filt or 'BIAS' in cal_type:
            if nmin_in_block <= filtered_df.loc[gb[key]].shape[0]:    # We only select execution blocks with more than nmin files
                exposure_list = []
                df_partition = filtered_df.loc[gb[key]]
                for index, row in df_partition.iterrows():
                    exposure_list.append(CalExp_WFI(row))
            
                #print(filtered_df.loc[gb[key]].shape[0])
                #display(filtered_df.loc[gb[key]])
                execution_block_dict[key[0]] = ExecutionBlock_WFI(exposure_list)

    # We should order the execution block list by distance to target date
    # ------------- TBD ----------------
    
    execution_block_delta_t = {}
    for akey in execution_block_dict.keys():
        delta_t = obs_date - datetime.strptime(akey, '%Y-%m-%dT%H:%M:%S')
        delta_days = abs(delta_t.total_seconds()/(3600.0*24.0))
        execution_block_delta_t[delta_days] = execution_block_dict[akey]
        if debug:
            print("delta_t [days]: {}".format(delta_days))

    # Here we create a dictionary indexed by the number of days
    #key_list = list(execution_block_delta_t.keys()).sort()
    key_list = [key for key in execution_block_delta_t.keys()]
    key_list.sort()
    if debug:
        print(key_list)
    execution_block_dict_sorted = {key:execution_block_delta_t[key] for key in key_list}
    
    # Here we access each partition data set to select a minimum of nagg_min frames
    frame_list = []
    for akey in execution_block_dict_sorted:
        exec_block=execution_block_dict_sorted[akey]
        for adataset in exec_block.data_setid_list:
            frame_list.append(adataset)
            if debug:
                print("{} {}    {} {} {} {} {}".format(akey, adataset, exec_block.tpl_start, exec_block.nexpo, exec_block.filter, exec_block.exptime_min, exec_block.exptime_max))
        if debug:        
            print("\n")
        if len(frame_list) >= nagg_min:
            break

    return frame_list
        

In [ ]:
# We select data according to time range of validity
#bias_archive_csv='wfi_bias_wdb_query_23900_eso.csv'    # more recent but with full set of keywords
bias_archive_csv='data/wfi_bias_wdb_query_24445_eso.csv'
domeff_archive_csv='data/wfi_domeff_wdb_query_25184_eso.csv'
skyff_archive_csv='data/wfi_skyff_wdb_query_29964_eso.csv'

obs_date = datetime.strptime('2002-03-11T00:22:04', '%Y-%m-%dT%H:%M:%S')
filt = 'BB#B/99_ESO842'
#skyff_date = datetime.strptime('2002-03-11T00:23:54', '%Y-%m-%dT%H:%M:%S')
#filt = 'BB#B/99_ESO842'
#filt = 'BB#V/89_ESO843'
#select_calibrations(bias_archive_csv, obs_date, filt, valid_days=2, nmin_in_block=10, cal_type='BIAS', time_key='TPL START')
#df_csv = pd.read_csv(domeff_archive_csv)
#selected_domeff_list = select_calibrations(df_csv, obs_date, filt, valid_days=3, nmin_in_block=10, nagg_min=10, cal_type='FLAT,SCREEN', time_key='TPL START', debug=True)
#print(selected_domeff_list)



In [ ]:
# ------------------ SELECTION PROCESS ----------------
# the science frames we would like reduced
raw_sci_dir = '/Users/fselman/Pictures/WFI/raw_orion'
raw_sci_files = glob.glob(raw_sci_dir+'/'+'**/WFI.*.fits', recursive=True)
raw_sci_files.sort()

# the archive csv files
bias_archive_csv='../data/wfi_bias_wdb_query_24445_eso.csv'
domeff_archive_csv='../data/wfi_domeff_wdb_query_25184_eso.csv'
skyff_archive_csv='../data/wfi_skyff_wdb_query_29964_eso.csv'

# here we determine the tags of the sci frames.
# Each tag is a tuple (TPL.START, INS.FILT1.NAME)
#
sciframe_tag_list = []
for afits in raw_sci_files:
    with fits.open(afits) as hdul:
        primary_hdr = hdul[0].header
        obs_date = datetime.strptime(primary_hdr['HIERARCH ESO TPL START'], '%Y-%m-%dT%H:%M:%S')
        filt = primary_hdr['HIERARCH ESO INS FILT1 NAME']
        sciframe_tag_list.append((obs_date, filt))
        #print(obs_date, filt)

        
#select_calibrations(df_csv, obs_date, filt, valid_days=3, nmin_in_block=10, nagg_min=10, \
#                    cal_type='FLAT,SCREEN', time_key='TPL START', debug=True)

# -- For each science frame
# ---- Select 10 bias frames within 1d or 2d
df_bias_csv = pd.read_csv(bias_archive_csv)
df_bias_csv = reform_df(df_bias_csv, 'TPL START')
bias_dataset_idlist = []
for atag in sciframe_tag_list:
    #print(atag[0], atag[1])
    obs_date = atag[0]
    filt = atag[1]
    bias_dataset_idlist += select_calibrations(df_bias_csv, obs_date, filt, valid_days=5, nmin_in_block=5, nagg_min=10, \
                    cal_type='BIAS', time_key='TPL START', debug=True)

unique_bias_list = list(set(bias_dataset_idlist))
unique_bias_list.sort()
print("bias_list legth: {}".format(len(bias_dataset_idlist)))  
print("bias_list legth: {}".format(len(unique_bias_list)))
#print(unique_bias_list)

# ---- Select 10 dome flat field frames within 1d or 2d
df_domeff_csv = pd.read_csv(domeff_archive_csv)
df_domeff_csv = reform_df(df_domeff_csv, 'TPL START')
domeff_dataset_idlist = []
for atag in sciframe_tag_list:
    #print(atag[0], atag[1])
    obs_date = atag[0]
    filt = atag[1]
    domeff_dataset_idlist += select_calibrations(df_domeff_csv, obs_date, filt, valid_days=30, nmin_in_block=5, nagg_min=10, \
                    cal_type='FLAT,SCREEN', time_key='TPL START', debug=True)

unique_dome_list = list(set(domeff_dataset_idlist))
unique_dome_list.sort()
print("dome_list legth:        {}".format(len(domeff_dataset_idlist)))  
print("unique dome_list legth: {}".format(len(unique_dome_list)))
#print(unique_dome_list)


# ---- Select 5 sky flat field frames within 1..30 days
df_skyff_csv = pd.read_csv(skyff_archive_csv)
df_skyff_csv = reform_df(df_skyff_csv, 'TPL START')
skyff_dataset_idlist = []
for atag in sciframe_tag_list:
    #print(atag[0], atag[1])
    obs_date = atag[0]
    filt = atag[1]
    skyff_dataset_idlist += select_calibrations(df_skyff_csv, obs_date, filt, valid_days=30, nmin_in_block=3, nagg_min=5, \
                    cal_type='FLAT,SKY', time_key='TPL START', debug=True)

unique_sky_list = list(set(skyff_dataset_idlist))
unique_sky_list.sort()
print("sky_list legth:        {}".format(len(skyff_dataset_idlist)))  
print("unique_sky_list legth: {}".format(len(unique_sky_list)))
#print(unique_sky_list)

# -- For each sky flat field group
# ---- Select 10 bias frames
for sky_frame in unique_sky_list:
    obs_date = datetime.strptime(sky_frame[4:-4],  '%Y-%m-%dT%H:%M:%S')
    filt = 'WHITE'
    bias_dataset_idlist += select_calibrations(df_bias_csv, obs_date, filt, valid_days=5, nmin_in_block=5, nagg_min=10, \
                    cal_type='BIAS', time_key='TPL START', debug=True)

unique_bias_list = list(set(bias_dataset_idlist))
unique_bias_list.sort()
print("bias_list legth: {}".format(len(bias_dataset_idlist)))  
print("bias_list legth: {}".format(len(unique_bias_list)))
#print(unique_bias_list)

# We create the calibration set that needs to be retrieved
calibration_set = unique_bias_list + unique_dome_list + unique_sky_list
calibration_set.sort()
for calframe in calibration_set:
    print(calframe)
